# 1. root 테스트

In [1]:
import requests

fastapi_url = "http://localhost:8080"

response = requests.get(fastapi_url)
print(response.status_code)

200


In [2]:
print(response.json())

{'message': 'Hello World'}


# 2. upload_image 테스트

In [3]:
upload_image_url = f"{fastapi_url}/upload_image"
print(upload_image_url)

http://localhost:8080/upload_image


In [ ]:
with open("../images/20260317173910-cat_test03.jpg", "rb") as f:
        # f는 이미지를 바이너리 형태로 불러오는 것이다. 
    files = {
        "file": f
    }

    response = requests.post(
    url = upload_image_url,
    files=files
    )

    if response.status_code == 200:
        print(response.json())

{'message': '이미지를 저장했습니다.', 'filename': '../images/20260318153222-20260317173910-cat_test03.jpg', 'time': '20260318153222'}


In [ ]:
# 만약 Streamlit에서 File_uploader로 이미지를 받은 상태라면?
# uploaded_file = st.file_uploader(....)
# uploaded_file.name : 파일 이름
# uploaded_file.getvalue() : 위에서 f와 동일
# uploaded_file.type : 파일 타입(png, jpg, ....)

# files = {
#     "file": (
#         uploaded_file.name,
#         uploaded_file.getvalue(),
#         uploaded_file.type
#     )
# }

In [ ]:
# 오늘의 과제 : 
# - request_test.ipynb 파일에 object detection 모델 예측 요청하기 (/detect_image)
# - (도전) streamlit에 object detection url 요청 반영해보기
# - (대체) file_uploader로 받은 변수 이름을 uploaded_file이라고 하자.
# -       

In [6]:
detect_img_url = f"{fastapi_url}/detect_image"
print(detect_img_url)

http://localhost:8080/detect_image


In [ ]:
with open("../images/20260317173910-cat_test03.jpg", "rb") as f:
    files = {
        "file": f
    }

    response = requests.post(
    url = detect_img_url,
    files=files
    )

    if response.status_code == 200:
        print(response.json())

{'message': '이미지를 저장했습니다.', 'filename': '../images/20260318155130-20260317173910-cat_test03.jpg', 'time': '20260318155130', 'object_detection': [{'box': [143.92433166503906, 19.968929290771484, 577.0, 490.9036560058594], 'confidence': {}, 'label': 'cat'}]}


# 3. detect_image 테스트

In [5]:
# 1. URL: 어디로 보내야 하나요?
object_detection_url = f"{fastapi_url}/detect_image"
print(object_detection_url)

# 2. data: 뭘 보내야 하나요?
## 통신방식: post
## 파일을 보내야 한다. -> 이미지 파일 
## files = { "file": f}
## 이미지 불러오는 방법
with open("./cat_test03.jpg", "rb") as f:
    files = { "file": f }
    print(files)

http://localhost:8080/detect_image
{'file': <_io.BufferedReader name='./cat_test03.jpg'>}


In [ ]:
import requests 

## 이미지 불러오는 방법
with open("./cat_test03.jpg", "rb") as f:
    files = { "file": f }

    response = requests.post(
        url=object_detection_url,
        files={ "file": f }
    )

    if response.status_code == 200:
        print(response.json())

### 과제

In [2]:
similarity_url = f"{fastapi_url}/similarity"
print(similarity_url)

http://localhost:8080/similarity


In [6]:
with open("../images/cat.jpg", "rb") as f:
    files = {
        "file": f
    }

    response = requests.post(
    url = similarity_url,
    files=files
    )

    if response.status_code == 200:
        print(response.json())

{'results': [{'label': 'a photo of a dog', 'probability': 0.014021211303770542}, {'label': 'a photo of a cat', 'probability': 0.985395610332489}, {'label': 'a photo of a pig', 'probability': 0.0005831099697388709}]}


# 4. Segmentation 테스트

In [1]:
import requests

# /segmentation 엔드포인트 URL 설정
segmentation_url = f"{fastapi_url}/segmentation"
print(segmentation_url)

# 이미지 파일을 바이너리로 열어 서버로 전송
with open("./sit.jpg", "rb") as f:
    files = {"file": f}

    response = requests.post(
        url=segmentation_url,
        files=files
    )

if response.status_code == 200:
    result = response.json()
    print(f"저장 파일명: {result['filename']}")
    print(f"탐지된 객체 수: {len(result['detections'])}개")
    print()

    # 탐지된 각 객체 정보 출력 (마스크 문자열은 너무 길어서 앞 30자만 표시)
    for i, det in enumerate(result["detections"]):
        print(f"[{i+1}번 객체]")
        print(f"  라벨     : {det['label']}")
        print(f"  신뢰도   : {det['confidence']:.2%}")
        print(f"  박스좌표 : {[round(v, 1) for v in det['box']]}")
        if det["mask"]:
            print(f"  마스크   : {det['mask'][:30]}... (base64 문자열)")  # 앞 30자만 미리보기
        else:
            print(f"  마스크   : None")
else:
    print(f"에러 발생: {response.status_code}")

NameError: name 'fastapi_url' is not defined

In [ ]:
import base64
import io
from PIL import Image
import matplotlib.pyplot as plt

# 서버에서 받은 base64 마스크를 이미지로 복원해서 시각화
detections = result["detections"]

# 마스크가 있는 객체만 필터링
mask_detections = [det for det in detections if det["mask"] is not None]
print(f"마스크가 있는 객체 수: {len(mask_detections)}개")

if mask_detections:
    fig, axes = plt.subplots(1, len(mask_detections), figsize=(5 * len(mask_detections), 5))

    # 객체가 1개일 때 axes가 리스트가 아니므로 리스트로 변환
    if len(mask_detections) == 1:
        axes = [axes]

    for ax, det in zip(axes, mask_detections):
        # base64 문자열 → 바이너리 → PIL Image로 복원
        mask_bytes = base64.b64decode(det["mask"])  # base64 디코딩 → PNG 바이너리
        mask_img = Image.open(io.BytesIO(mask_bytes))  # PNG 바이너리 → PIL Image

        ax.imshow(mask_img, cmap="gray")  # 흑백으로 표시 (0=배경, 255=객체)
        ax.set_title(f"{det['label']} ({det['confidence']:.0%})")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

# 5. Clip 테스트

In [ ]:
import requests

# /clip 엔드포인트 URL 설정
clip_url = f"{fastapi_url}/clip"
print(clip_url)

# 이미지 파일을 바이너리로 열어 서버로 전송
with open("./sit.jpg", "rb") as f:
    files = {"file": f}

    response = requests.post(
        url=clip_url,
        files=files
    )

if response.status_code == 200:
    result = response.json()

    print(f"예측 결과: {result['predicted']}")  # 가장 유사도 높은 카테고리
    print()

    # 각 카테고리별 유사도 출력 (내림차순 정렬)
    print("=== 카테고리별 유사도 ===")
    sorted_sim = sorted(result["similarities"].items(), key=lambda x: x[1], reverse=True)
    for label, prob in sorted_sim:
        print(f"  {label:5s}: {prob:.1%}")
else:
    print(f"에러 발생: {response.status_code}")

# 6. 챗봇 테스트

In [1]:
import requests

chat_url = "http://localhost:8080/chat"

data = {
    "message": "안녕하세요?"
}

response = requests.post(
    url = chat_url,
    json=data
)
if response.status_code == 200:
    print(response.json())

{'text': '안녕하세요! 반가워요. 무엇을 도와드릴까요?\n\n- 질문에 답하기\n- 글쓰기나 아이디어 다듬기\n- 번역이나 학습 도움\n- 코딩/기술 문의\n- 요리 레시피나 여행 정보 등\n\n원하시는 주제를 말씀해주시면 바로 도와드릴게요.'}
